# Suggested walks pipeline: street network to GeoJSON

This notebook makes CatMap's **suggested walks**: loops of about 2, 3 and 5 km from three starting points in the Plateau, each in two themes.

- 🐈 **Cat walk**: passes as many of my cat sightings as possible.
- 🌿 **Nature walk**: uses as much of the green alleys as possible, and passes community gardens where it can.

```
OpenStreetMap (downloaded by OSMnx)  -> data/plateau_walk.graphml   the walking network, saved so it downloads once
data/cat_photos.geojson, ruelles_vertes.geojson, communal_gardens.geojson, water_fountains.geojson
                                     -> data/walks.geojson          output: 18 walks (3 starts x 3 lengths x 2 themes)
```

Run each cell in order, top to bottom. The whole notebook takes about a minute, plus about a minute more the first time to download the network.

## 0) One-time setup
**OSMnx** downloads street networks from OpenStreetMap and finds routes on them. It needs **SciPy** to find the nearest street corner to a point. Run this once, then restart the kernel.

In [ ]:
%pip install osmnx scipy

  Using cached osmnx-2.1.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached scipy-1.18.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.2-cp314-cp314-macosx_10_15_universal2.whl.metadata (46 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
Using cached osmnx-2.1.1-py3-none-any.whl (104 kB)
Using cached scipy-1.18.1-cp314-cp314-macosx_14_0_arm64.whl (20.5 MB)
Using cached networkx-3.7-py3-none-any.whl (2.1 MB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.2-cp314-cp314-macosx_10_15_universal2.whl (367 kB)
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [osmnx]32m2/7 [networkx]
Note: you may need to

## 1) Import libraries and settings
All distances are worked out in metres, in Montréal's **MTM zone 8** projection (`EPSG:32188`, the same one as Lab 4's Recipe 2). The output is converted back to longitude and latitude at the end.

In [2]:
import os
import random
import networkx as nx
import osmnx as ox
import geopandas as gpd
from shapely.geometry import LineString, Point

MTM = "EPSG:32188"                       # metres, for measuring
GRAPH_PATH = "../data/plateau_walk.graphml"
OUT_PATH = "../data/walks.geojson"

STARTS = {                               # name: (latitude, longitude)
    "Métro Mont-Royal": (45.5240948, -73.5830385),
    "Parc Laurier": (45.5320804, -73.5878482),
    "Métro Laurier": (45.5269788, -73.5892582),
}
LENGTHS_KM = [2, 3, 5]
TRIES = 300          # candidate loops tried for each start, length and theme
TOLERANCE = 0.15     # a "3 km" walk can be 2.55 to 3.45 km
MAX_REPEAT = 0.25    # at most a quarter of a walk can go back over streets it already used
NEAR_M = 50          # a cat, garden or fountain within 50 m of the route counts as "passed"
random.seed(464)     # same random choices every run, so the walks don't change each time
ox.settings.use_cache = False   # no extra cache folder: the network is saved as a .graphml instead

## 2) Download the walking network, or load the saved copy
`graph_from_polygon` downloads every path you can walk on inside an area: sidewalks, footpaths, park paths, pedestrian streets and **alleys**. `network_type="walk"` leaves out highways and other roads people can't walk on. The area is the Plateau-Mont-Royal borough (which includes Mile End) plus 500 m all round, so walks near the edge aren't cut off.

The network is a **graph**: street corners are *nodes* and the street pieces between them are *edges*, each with its length in metres.

In [3]:
if os.path.exists(GRAPH_PATH):
    G = ox.load_graphml(GRAPH_PATH)
    print("Loaded the saved network from", GRAPH_PATH)
else:
    print("Downloading the walking network from OpenStreetMap (about a minute)...")
    area = ox.geocode_to_gdf("Le Plateau-Mont-Royal, Montréal, Québec, Canada")
    area = area.to_crs(MTM).buffer(500).to_crs("EPSG:4326")        # add 500 m all round
    G = ox.graph_from_polygon(area.iloc[0], network_type="walk")
    ox.save_graphml(G, GRAPH_PATH)
    print("Saved to", GRAPH_PATH)

G = ox.project_graph(G, to_crs=MTM)        # coordinates in metres
G = ox.convert.to_undirected(G)            # walking works both ways on every street
print(len(G.nodes), "street corners,", len(G.edges), "street pieces")

Loaded the saved network from ../data/plateau_walk.graphml
13372 street corners, 19080 street pieces


## 3) Load the map layers and mark the green alleys on the network
A street piece counts as **green** if more than half of it lies inside a green alley (with 8 m of slack, since the alley shapes and the OpenStreetMap lines don't line up exactly).

For nature walks, the route-finder is told that green pieces are **"cheaper" to walk**: they count as 40% of their real length, so the shortest route prefers going through green alleys.

In [4]:
cats = gpd.read_file("../data/cat_photos.geojson").to_crs(MTM)
alleys = gpd.read_file("../data/ruelles_vertes.geojson").to_crs(MTM)
gardens = gpd.read_file("../data/communal_gardens.geojson").to_crs(MTM)
fountains = gpd.read_file("../data/water_fountains.geojson").to_crs(MTM)

edges = ox.graph_to_gdfs(G, nodes=False)
alley_area = alleys.buffer(8).union_all()
inside = edges.geometry.intersection(alley_area).length
edges["green_m"] = inside.where(inside > 0.5 * edges.geometry.length, 0)
nx.set_edge_attributes(G, edges["green_m"].to_dict(), "green_m")

for u, v, k, d in G.edges(keys=True, data=True):
    d["nature_cost"] = d["length"] - 0.6 * d["green_m"]

print(round(edges["green_m"].sum() / 1000, 1), "km of the network runs through green alleys")

# The street corner nearest each cat and each garden, used to steer walks towards them
cat_nodes = set(ox.distance.nearest_nodes(G, cats.geometry.x, cats.geometry.y))
garden_nodes = set(ox.distance.nearest_nodes(G, gardens.geometry.x, gardens.geometry.y))

23.2 km of the network runs through green alleys


## 4) Helper functions
- `edge(u, v)`: the street piece between two corners.
- `route_line(path)`: turns a list of corners into one line that follows the real street shapes.
- `make_loop(start, a, b, weight)`: a loop **start → a → b → start**, using the shortest route for each leg. Picking `a` and `b` about a third of the walk's length away from the start gives a loop of roughly the right length.
- `score(path)`: measures a loop: its length, how much of it doubles back, and the cats, green alley metres, gardens and fountains along it.

In [5]:
def edge(u, v):
    # there can be more than one street piece between two corners: take the shortest
    return min(G.get_edge_data(u, v).values(), key=lambda d: d["length"])


def route_line(path):
    coords = [(G.nodes[path[0]]["x"], G.nodes[path[0]]["y"])]
    for u, v in zip(path[:-1], path[1:]):
        data = edge(u, v)
        if "geometry" in data:
            piece = list(data["geometry"].coords)
            if piece[0] != (G.nodes[u]["x"], G.nodes[u]["y"]):
                piece = piece[::-1]          # make the piece run in the walking direction
        else:
            piece = [(G.nodes[u]["x"], G.nodes[u]["y"]), (G.nodes[v]["x"], G.nodes[v]["y"])]
        coords.extend(piece[1:])
    return LineString(coords)


def make_loop(start, a, b, weight):
    try:
        leg1 = nx.shortest_path(G, start, a, weight=weight)
        leg2 = nx.shortest_path(G, a, b, weight=weight)
        leg3 = nx.shortest_path(G, b, start, weight=weight)
    except nx.NetworkXNoPath:
        return None
    return leg1 + leg2[1:] + leg3[1:]


def score(path):
    steps = list(zip(path[:-1], path[1:]))
    length = sum(edge(u, v)["length"] for u, v in steps)
    unique = {tuple(sorted(step, key=str)) for step in steps}     # each street piece once
    line = route_line(path)
    near = line.buffer(NEAR_M)
    return {
        "geometry": line,
        "length_m": length,
        "repeat": 1 - sum(edge(u, v)["length"] for u, v in unique) / length,
        "cats": int(cats.within(near).sum()),
        "green_m": sum(edge(u, v)["green_m"] for u, v in unique),
        "gardens": int(gardens.within(near).sum()),
        "fountains": int(fountains.within(near).sum()),
    }

## 5) Try many loops, keep the best one for each start, length and theme
For each start and length:
1. Find every corner whose walking distance from the start is about a third of the walk's length (the *ring*).
2. Try 300 loops through two corners from the ring. **Cat walks** usually pick corners next to cats; **nature walks** often pick one next to a garden, and route through green alleys.
3. Throw away loops that are too short, too long, or double back too much.
4. Keep the best: **cat walks** by number of cats (green alley metres break ties); **nature walks** by green alley metres, plus 300 m for each garden passed.

In [6]:
rows = []
for name, (lat, lon) in STARTS.items():
    x, y = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(MTM).iloc[0].coords[0]
    start = ox.distance.nearest_nodes(G, x, y)

    for km in LENGTHS_KM:
        target = km * 1000
        dist = nx.single_source_dijkstra_path_length(G, start, cutoff=target / 2, weight="length")
        ring = [n for n, d in dist.items() if 0.22 * target <= d <= 0.38 * target]
        wide = [n for n, d in dist.items() if 0.15 * target <= d <= 0.45 * target]
        ring_cats = [n for n in wide if n in cat_nodes]
        ring_gardens = [n for n in wide if n in garden_nodes]

        for theme in ["cats", "nature"]:
            best, best_value = None, -1
            for _ in range(TRIES):
                if theme == "cats" and len(ring_cats) >= 2 and random.random() < 0.7:
                    a, b = random.sample(ring_cats, 2)
                elif theme == "nature" and ring_gardens and random.random() < 0.5:
                    a, b = random.choice(ring_gardens), random.choice(ring)
                else:
                    a, b = random.sample(ring, 2)
                path = make_loop(start, a, b, "length" if theme == "cats" else "nature_cost")
                if path is None:
                    continue
                s = score(path)
                if abs(s["length_m"] - target) > TOLERANCE * target or s["repeat"] > MAX_REPEAT:
                    continue
                if theme == "cats":
                    value = s["cats"] * 100 + s["green_m"] / 10
                else:
                    value = s["green_m"] + 300 * s["gardens"]
                if value > best_value:
                    best, best_value = s, value

            rows.append({"start": name, "km": km, "theme": theme, **best})
            print(f"{name:17} {km} km {theme:6}  {best['length_m'] / 1000:.2f} km, {best['cats']:2} cats, "
                  f"{best['green_m']:5.0f} m green alleys, {best['gardens']} gardens, {best['fountains']} fountains")

Métro Mont-Royal  2 km cats    2.06 km, 21 cats,     0 m green alleys, 0 gardens, 2 fountains
Métro Mont-Royal  2 km nature  2.26 km,  4 cats,   912 m green alleys, 0 gardens, 1 fountains
Métro Mont-Royal  3 km cats    2.95 km, 24 cats,     7 m green alleys, 0 gardens, 2 fountains
Métro Mont-Royal  3 km nature  3.13 km,  1 cats,  1549 m green alleys, 0 gardens, 1 fountains
Métro Mont-Royal  5 km cats    4.92 km, 27 cats,     6 m green alleys, 0 gardens, 5 fountains
Métro Mont-Royal  5 km nature  5.09 km,  1 cats,  2071 m green alleys, 1 gardens, 1 fountains
Parc Laurier      2 km cats    1.77 km,  1 cats,   292 m green alleys, 0 gardens, 2 fountains
Parc Laurier      2 km nature  2.14 km,  0 cats,   938 m green alleys, 0 gardens, 2 fountains
Parc Laurier      3 km cats    3.14 km,  7 cats,   203 m green alleys, 0 gardens, 3 fountains
Parc Laurier      3 km nature  3.21 km,  1 cats,  1540 m green alleys, 0 gardens, 2 fountains
Parc Laurier      5 km cats    4.84 km, 27 cats,     6 m gre

## 6) Write the walks
One line per walk, with what the website's walk panel shows. **Output:** `../data/walks.geojson`

In [7]:
walks = gpd.GeoDataFrame(rows, geometry="geometry", crs=MTM)
walks["length_km"] = (walks["length_m"] / 1000).round(2)
walks["green_m"] = walks["green_m"].round(0).astype(int)
walks = walks[["start", "km", "theme", "length_km", "cats", "green_m", "gardens", "fountains", "geometry"]]

walks.to_crs("EPSG:4326").to_file(OUT_PATH, driver="GeoJSON", COORDINATE_PRECISION=6)
print("Wrote", OUT_PATH, "-", round(os.path.getsize(OUT_PATH) / 1_000_000, 2), "MB")
walks.drop(columns="geometry")

Wrote ../data/walks.geojson - 0.14 MB


,start,km,theme,length_km,cats,green_m,gardens,fountains
0,Métro Mont-Royal,2,cats,2.06,21,0,0,2
1,Métro Mont-Royal,2,nature,2.26,4,912,0,1
2,Métro Mont-Royal,3,cats,2.95,24,7,0,2
3,Métro Mont-Royal,3,nature,3.13,1,1549,0,1
4,Métro Mont-Royal,5,cats,4.92,27,6,0,5
5,Métro Mont-Royal,5,nature,5.09,1,2071,1,1
6,Parc Laurier,2,cats,1.77,1,292,0,2
7,Parc Laurier,2,nature,2.14,0,938,0,2
8,Parc Laurier,3,cats,3.14,7,203,0,3
9,Parc Laurier,3,nature,3.21,1,1540,0,2


## 7) Check the file
The check cell from Lab 4, with the four settings changed for this file. 3 starts x 3 lengths x 2 themes = 18 walks.

In [8]:
# ---- settings: change these for your own data ----
FILE = OUT_PATH
WEST, EAST, SOUTH, NORTH = -73.63, -73.54, 45.50, 45.55   # a box around the Plateau, in degrees
EXPECTED = 18             # how many features you expect (rows after your filters)
POPUP_FIELDS = ["start", "km", "theme", "length_km", "cats", "green_m", "gardens", "fountains"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some features are outside your box: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow. Filter or simplify.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['LineString']
All checks passed


## 8) Next
`data/walks.geojson` is ready for the website's "Suggested walks" panel (choose a start, a length and a theme; the walk lights up on the map with its numbers). To change the starting points or lengths, edit `STARTS` and `LENGTHS_KM` in step 1 and run everything again; the saved network is reused, so it only takes about a minute.